In [29]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/P06'


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [30]:
import os
import zipfile
import shutil
import pandas as pd

# Define paths
raw_dir = os.path.join(BASE, 'data/raw')
token_path = os.path.join(raw_dir, 'Flickr8k.token.txt')
zip_drive_path = os.path.join(raw_dir, 'Flickr8k_text.zip')
zip_local_path = '/content/Flickr8k_text.zip'

# Ensure the raw directory exists
os.makedirs(raw_dir, exist_ok=True)

# Extract zip if token file is missing
if not os.path.exists(token_path):
    # Check if the zip file is in Google Drive or local runtime
    if os.path.exists(zip_drive_path):
        print(f"Extracting {zip_drive_path} to {raw_dir}...")
        with zipfile.ZipFile(zip_drive_path, 'r') as zip_ref:
            zip_ref.extractall(raw_dir)
        print("Extraction complete!")
    elif os.path.exists(zip_local_path):
        print(f"Copying and extracting local {zip_local_path} to {raw_dir}...")
        shutil.copy(zip_local_path, zip_drive_path)
        with zipfile.ZipFile(zip_drive_path, 'r') as zip_ref:
            zip_ref.extractall(raw_dir)
        print("Extraction complete!")
    else:
        raise FileNotFoundError(f"Could not find 'Flickr8k.token.txt' or 'Flickr8k_text.zip' in {raw_dir}.")

# Process the extracted text data
rows = []
for line in open(token_path, encoding='utf8'):
    image, caption = line.rstrip().split('\t', 1)
    rows.append((image.split('#')[0], caption))

captions = pd.DataFrame(rows, columns=['image_id', 'caption'])
print(captions.head())
print(len(captions))
print(captions.groupby('image_id').size().value_counts())

                    image_id  \
0  1000268201_693b08cb0e.jpg   
1  1000268201_693b08cb0e.jpg   
2  1000268201_693b08cb0e.jpg   
3  1000268201_693b08cb0e.jpg   
4  1000268201_693b08cb0e.jpg   

                                             caption  
0  A child in a pink dress is climbing up a set o...  
1              A girl going into a wooden building .  
2   A little girl climbing into a wooden playhouse .  
3  A little girl climbing the stairs to her playh...  
4  A little girl in a pink dress going into a woo...  
40460
5    8092
Name: count, dtype: int64


In [31]:
import random
random.seed(42)
docs = []
for image_id, group in captions.groupby('image_id'):
    caps = list(group['caption'])
    q = random.choice(caps)
    caps.remove(q)
    docs.append({'image_id': image_id, 'doc_text': ' '.join(caps), 'query_text': q})
corpus = pd.DataFrame(docs)
print(len(corpus))
print(corpus.iloc[0].to_dict())

8092
{'image_id': '1000268201_693b08cb0e.jpg', 'doc_text': 'A girl going into a wooden building . A little girl climbing into a wooden playhouse . A little girl climbing the stairs to her playhouse . A little girl in a pink dress going into a wooden cabin .', 'query_text': 'A child in a pink dress is climbing up a set of stairs in an entry way .'}


In [32]:
same = sum(r.query_text in r.doc_text for r in corpus.itertuples())
print(same)


10


In [33]:
import re, unicodedata, nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer, WordNetLemmatizer

stop_words = set(stopwords.words('english'))
stemmer = SnowballStemmer('english')
lemmatizer = WordNetLemmatizer()

def preprocess(text, variant='base'):
    text = unicodedata.normalize('NFKC', text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    tokens = text.split()
    if variant in ('stop', 'stem', 'lemma'):
        tokens = [t for t in tokens if t not in stop_words]
    if variant == 'stem':
        tokens = [stemmer.stem(t) for t in tokens]
    if variant == 'lemma':
        tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return tokens

for v in ['base', 'stop', 'stem', 'lemma']:
    print(v, preprocess('Two Children are running on the beach!', v))

base ['two', 'children', 'are', 'running', 'on', 'the', 'beach']
stop ['two', 'children', 'running', 'beach']
stem ['two', 'children', 'run', 'beach']
lemma ['two', 'child', 'running', 'beach']


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [34]:
for v in ['base', 'stop', 'stem', 'lemma']:
    vocab = set()
    for text in corpus['doc_text']:
        vocab.update(preprocess(text, v))
    print(v, len(vocab))

base 7737
stop 7620
stem 5167
lemma 6529


In [35]:
for w in ['children', 'kids', 'bicycle', 'bicycles', 'running', 'run']:
    print(w, '| stem:', stemmer.stem(w),
          '| lemma as noun:', lemmatizer.lemmatize(w),
          '| lemma as verb:', lemmatizer.lemmatize(w, pos='v'))

children | stem: children | lemma as noun: child | lemma as verb: children
kids | stem: kid | lemma as noun: kid | lemma as verb: kid
bicycle | stem: bicycl | lemma as noun: bicycle | lemma as verb: bicycle
bicycles | stem: bicycl | lemma as noun: bicycle | lemma as verb: bicycle
running | stem: run | lemma as noun: running | lemma as verb: run
run | stem: run | lemma as noun: run | lemma as verb: run


In [36]:
ids = corpus['image_id'].sample(frac=1, random_state=42).tolist()
n = len(ids)
split = {}
for i, img in enumerate(ids):
    split[img] = 'train' if i < 0.7 * n else ('val' if i < 0.85 * n else 'test')
corpus['split'] = corpus['image_id'].map(split)
print(corpus['split'].value_counts())

corpus.to_csv(BASE + '/outputs/cleaned_corpus.csv', index=False)

qrels = corpus[['image_id', 'query_text']].copy()
qrels.insert(0, 'query_id', ['q' + str(i) for i in range(len(qrels))])
qrels = qrels.rename(columns={'image_id': 'relevant_image_id'})
qrels.to_csv(BASE + '/outputs/queries_qrels.csv', index=False)

split
train    5665
val      1214
test     1213
Name: count, dtype: int64


In [37]:
ids = corpus['image_id'].sample(frac=1, random_state=42).tolist()
n = len(ids)
split = {}
for i, img in enumerate(ids):
    split[img] = 'train' if i < 0.7 * n else ('val' if i < 0.85 * n else 'test')
corpus['split'] = corpus['image_id'].map(split)
print(corpus['split'].value_counts())

corpus.to_csv(BASE + '/outputs/cleaned_corpus.csv', index=False)

qrels = corpus[['image_id', 'query_text']].copy()
qrels.insert(0, 'query_id', ['q' + str(i) for i in range(len(qrels))])
qrels = qrels.rename(columns={'image_id': 'relevant_image_id'})
qrels.to_csv(BASE + '/outputs/queries_qrels.csv', index=False)

split
train    5665
val      1214
test     1213
Name: count, dtype: int64


In [38]:
import pandas as pd
check = pd.read_csv(BASE + '/outputs/cleaned_corpus.csv')
print(check.shape)
print(check.columns.tolist())
q = pd.read_csv(BASE + '/outputs/queries_qrels.csv')
print(q.shape)
print(q.head(3))

(8092, 4)
['image_id', 'doc_text', 'query_text', 'split']
(8092, 3)
  query_id          relevant_image_id  \
0       q0  1000268201_693b08cb0e.jpg   
1       q1  1001773457_577c3a7d70.jpg   
2       q2  1002674143_1b742ab4b8.jpg   

                                          query_text  
0  A child in a pink dress is climbing up a set o...  
1         A black dog and a spotted dog are fighting  
2  A small girl in the grass plays with fingerpai...  


In [39]:
check = pd.read_csv(BASE + '/outputs/cleaned_corpus.csv')
print(check.shape, check.columns.tolist())

(8092, 4) ['image_id', 'doc_text', 'query_text', 'split']


In [ ]:
from google.colab import drive
drive.mount('/content/drive')